# Accelerating Drug Discovery with ceSAR: Combining Transcriptional Signatures and AI-Powered Docking

> **Paper**: [Accelerating drug discovery and repurposing by combining transcriptional signature connectivity with docking](https://www.science.org/doi/10.1126/sciadv.adj3010)  
> **Authors**: Thorman et al.  
> **Published**: Science Advances, August 2024  
> **Code**: Official ceSAR implementation at [sig2lead GitHub](https://github.com/sig2lead)

## Introduction

Drug discovery remains one of the most challenging and time-consuming endeavors in biomedical research, often taking over a decade and billions of dollars to bring a single compound from initial screening to clinical approval. The ability to rapidly identify promising drug candidates—whether for new therapeutic targets or for repurposing existing drugs—has become paramount, especially in the face of public health crises like the COVID-19 pandemic. Yet traditional approaches face two critical bottlenecks: the computational expense of virtual screening across vast chemical libraries, and the high false-positive rates that plague even state-of-the-art docking methods.

Enter **ceSAR** (connectivity enhanced Structure Activity Relationship), an innovative computational framework that promises to revolutionize how we identify drug candidates. By elegantly combining three powerful principles—transcriptional signature concordance, ultrafast chemical similarity search, and biophysical docking simulations—ceSAR achieves something remarkable: it reduces false-positive rates by 3-4 fold while simultaneously cutting computational costs by several orders of magnitude. This isn't just an incremental improvement; it's a fundamental rethinking of the virtual screening pipeline that makes large-scale drug discovery accessible to researchers without massive computational infrastructure.

### The Evolution of Drug Discovery: From Phenotypic Screens to Systems Pharmacology

To truly appreciate ceSAR's innovation, we need to trace the intellectual lineage of drug discovery approaches that led to this moment.

**The Early Days: Structure-Based Drug Design (1980s-1990s)**  
The story begins with the rise of structure-based drug design in the 1980s. The ability to determine three-dimensional protein structures through X-ray crystallography revolutionized medicinal chemistry, enabling researchers to rationally design molecules that complemented binding pockets. Early docking algorithms attempted to predict binding poses and affinities computationally, though with limited accuracy. The field was constrained by computing power—screening even modest libraries required significant computational resources.

**The Genomics Revolution (2000s)**  
The completion of the Human Genome Project and subsequent advances in high-throughput screening created new opportunities. Large pharmaceutical companies built libraries of hundreds of thousands of compounds and developed robotic systems for phenotypic screening. But this brute-force approach was expensive and often yielded hits whose mechanisms of action remained mysterious.

**The Connectivity Map Era (2006)**  
A watershed moment came in 2006 when Lamb et al. introduced the Connectivity Map concept in Science. Their insight was profound: transcriptional responses encode information about drug mechanisms and could be used to connect diseases with potential treatments. If a disease causes certain genes to be up-regulated and others down-regulated, perhaps a drug that induces the opposite pattern could reverse the disease phenotype. This "signature connectivity" approach opened entirely new avenues for drug repurposing.

**LINCS: Scaling Up Pharmacogenomics (2017)**  
The LINCS consortium, announced in 2017, represented an unprecedented expansion of the Connectivity Map vision. By profiling over 15,000 small molecules and thousands of genetic perturbations across multiple cell lines, LINCS created a comprehensive map linking the "drug-like universe" with the "druggable genome." The development of iLINCS provided researchers with tools to explore these signatures and compute concordance scores, but a crucial limitation remained: LINCS only profiled specific compounds, not the millions of molecules researchers might want to screen.

**Structure Activity Relationships: The Chemical Similarity Principle (Ongoing)**  
Parallel to these developments, the principle of Structure-Activity Relationships (SAR) held that structurally similar molecules tend to have similar biological activities. If compound A binds to a target, compounds with similar chemical structures are more likely to bind as well. Chemical similarity searches using fingerprints—binary vectors encoding structural features—became standard tools in cheminformatics. The Tanimoto coefficient emerged as the gold standard for quantifying molecular similarity.

**Deep Learning Enters the Arena (2016-Present)**  
The deep learning revolution that transformed computer vision and natural language processing inevitably reached drug discovery. Methods like DeepVS and Deep Docking promised to learn patterns from databases of known active and inactive compounds, potentially outperforming physics-based docking. However, challenges of overfitting, dataset bias, and robust generalization have tempered initial optimism. The field continues to grapple with whether data-driven approaches can truly capture the complexity of molecular interactions.

### The Problem: When Biology Meets Chemistry

Imagine you're searching for inhibitors of a specific protein target—let's say BCL2A1, an antiapoptotic protein implicated in melanoma and inflammation. Traditional docking approaches might screen 20,000 compounds, attempting to predict which molecules will physically fit into the protein's binding pocket. But here's the catch: docking simulations, while powerful, often struggle to correctly rank true binders among look-alike decoys. They might successfully eliminate the most obvious non-binders, but as you narrow down to the top candidates, the signal gets noisy. You could end up with a "catastrophic failure"—spending weeks on experimental validation only to find that none of your top 100 computational hits actually work.

On the other hand, transcriptional profiling offers a completely different lens. The landmark LINCS (Library of Integrated Network-based Cellular Signatures) project has profiled the transcriptional responses of over 15,000 drug-like molecules and ~4,400 gene knockdowns across multiple cell lines. This massive resource captures how cells respond when you knock down a gene or add a small molecule. The intuition is elegant: if a small molecule truly inhibits protein X, its transcriptional signature should resemble what happens when you knock down gene X.

But transcriptional concordance alone has limitations too. Similar downstream signatures might result from inhibiting different proteins in the same pathway. If you're looking for SRC inhibitors, you might also identify compounds that target EGFR or JUN—all part of the same EGFR-SRC-JUN signaling cascade. They're pathway inhibitors, but not necessarily direct binders to your target of interest.

### The Solution: Best of Both Worlds

ceSAR's breakthrough lies in recognizing that these two approaches—transcriptional concordance and biophysical docking—are beautifully complementary. They fail in different ways, which means combining them can dramatically reduce false positives while preserving true hits. The method works in an intelligent two-stage pipeline:

**Stage 1: Signature-Guided Filtering (ceSAR-S)**  
For each candidate molecule in your library, ceSAR computes its chemical similarity to LINCS compounds whose transcriptional signatures are "concordant" with your target gene's knockdown signature. This similarity score becomes your initial ranking. The beauty here is that this step is structure-independent—you don't need a crystal structure of your target protein. Even better, through an ingenious algorithm called **minSim**, this step runs about 50,000× faster than traditional docking, reducing library screening from weeks to mere minutes on a laptop.

**Stage 2: Docking-Based Refinement**  
The top-ranked candidates from Stage 1 (typically 1-5% of the library) are then subjected to molecular docking simulations. Now you're asking: "Among these compounds that transcriptionally mimic target knockdown, which ones can actually fit into the binding pocket?" By combining rankings from both stages—either through simple geometric averaging or through machine learning consensus—ceSAR identifies candidates that satisfy both biological plausibility (signature concordance) and physical compatibility (binding affinity).

The results speak for themselves. On the widely-used DUD-E benchmark of 20 diverse protein targets, ceSAR achieves ~30-40% precision at 0.1% library size—meaning roughly 3-4 out of every 10 top-ranked compounds are true binders. Compare this to ~10% for molecular docking alone, and you understand why this represents a paradigm shift.

### Understanding ceSAR's Core Concepts: An Intuitive View

Let's demystify the key concepts that make ceSAR work:

**Transcriptional Signatures as Molecular Fingerprints**  
Think of a transcriptional signature as a cellular "selfie"—a snapshot of how thousands of genes respond to a perturbation. When you knock down a gene or add a drug, cells adjust their gene expression programs. Measure mRNA levels for ~1,000 landmark genes, and you capture the essence of the cellular response. The LINCS signatures are these high-dimensional vectors of gene expression changes.

**Concordance: The Biological Phenocopy Principle**  
Two signatures are "concordant" if they're positively correlated—genes that go up in one signature tend to go up in the other, and vice versa. If knocking down gene X causes certain genes to increase and others to decrease, and adding compound Y produces a similar pattern, that's evidence that Y might inhibit protein X. The extreme Pearson correlation coefficient quantifies this relationship, with statistical significance determined by Bonferroni correction.

**Chemical Similarity: The Neighborhood Principle**  
In chemistry, neighbors matter. The atom-pair fingerprint representation encodes which pairs of atoms (connected by various path lengths) exist in a molecule. Similar molecules share many of these features. The Tanimoto coefficient—the ratio of shared features to total features—quantifies similarity from 0 (nothing in common) to 1 (identical). Compounds with Tanimoto ≥ 0.8 are considered close analogs that likely share biological activities.

**The Transfer Learning Analogy**  
You can think of ceSAR as transfer learning for drug discovery. LINCS provides "pre-training" on 15,000 compounds. When you want to score a new compound, you find its nearest concordant neighbor in the LINCS space and transfer that concordance signal via chemical similarity. It's analogous to how word embeddings in NLP allow you to transfer semantic relationships to new words based on their proximity to known words.

**Consensus as Ensemble Learning**  
Just as ensemble methods in machine learning combine multiple models for robust predictions, ceSAR's consensus approaches combine evidence from different sources. The geometric mean of ranks balances signature-based and docking-based evidence. If a compound ranks 10th by signatures and 100th by docking, its consensus rank is ~32 (√(10×100))—a reasonable compromise that hedges against either method's weaknesses.


## Notebook Roadmap

### Learning Objectives

By the end of this comprehensive notebook series, you will:

- **Understand** the biological and computational foundations of signature connectivity analysis and how transcriptional phenocopying enables target-specific compound identification
- **Master** the ceSAR algorithm including the ultrafast minSim chemical similarity search and signature concordance scoring
- **Implement** the complete ceSAR pipeline from signature retrieval through consensus ranking using modern AI-powered docking
- **Reproduce** key results from the original paper including DUD-E benchmark analysis and BCL2A1 inhibitor discovery
- **Apply** ceSAR to your own targets of interest using custom libraries and interpret results critically
- **Analyze** performance trade-offs between speed and accuracy across different ceSAR variants and understand when each approach is most appropriate

---

### Sections Overview

#### 1. [Environment Setup](#environment-setup)
- Building and configuring the Docker container with all dependencies
- Installing and testing Boltz-2 for AI-powered docking


#### 2. [Data Acquisition and Preprocessing](#data-acquisition-and-preprocessing)
- Download LINCS datasets
  - LINCS L1000 Chemical Perturbations (2021): `170.8GB, 1805898x23614 profiles`
    - (wget https://lincs-dcic.s3.amazonaws.com/LINCS-data-2020/RNA-seq/cp_predicted_RNAseq_profiles.gctx)
  - LINCS L1000 shRNA Perturbations (2021): `42.86GB, 453175x23614 profiles`
    - (wget https://lincs-dcic.s3.amazonaws.com/LINCS-data-2020/RNA-seq/shRNA_predicted_RNAseq_profiles.gctx)
  - LINCS L1000 Mean Coefficients (2021): `~200MB`
    - wget https://lincs-dcic.s3.amazonaws.com/LINCS-sigs-2021/means/cp_mean_coeff_mat.tsv.gz
  - LINCS L1000 Metadata (2021): `~5MB`
    - wget https://s3.amazonaws.com/lincs-dcic/sigcom-lincs-metadata/LINCS_small_molecules.tsv
- Parse LINCS L1000 `.gctx` files using `cmapPy` or `h5py`
- Preparing compound libraries and generating molecular fingerprints
  - CHEMBL Drug Library: 4,000+ clinically tested compounds
    - (wget https://ftp.ebi.ac.uk/pub/databases/chembl/SureChEMBL/bulk_data/latest/compounds.parquet)
  - Natural product library: ~25,000 compounds
    - (wget https://ac-discovery.com/wp-content/uploads/MEGx_Release_2025_09_05.7z)
    - (wget https://ac-discovery.com/wp-content/uploads/NATx_Release_2025_09_05.zip)
    - (wget https://ac-discovery.com/wp-content/uploads/MACROx_Release_2025_09_05.7z)
  - Cyclic peptide library: ~8,000 compounds 
    - (wget https://www.biosino.org/iMAC/cyclicpepedia/static/data/excel/Bioassay.xlsx)
    - (wget https://www.biosino.org/iMAC/cyclicpepedia/static/data/structure_2dmol.zip)
  - Generative library from flowr.root (Pfizer) or drugflow.ai (VantAI)
    - (wget https://www.biosino.org/iMAC/cyclicpepedia/static/data/structure_2dmol.zip)

#### 3. [Signature Concordance Analysis](#signature-concordance-analysis)
- Extract PKC knockdown signature from shRNA dataset
  - Search for PKC isoforms: PRKCA, PRKCB, PRKCD, PRKCE, etc.
  - Generate consensus knockdown signature across cell lines
- Compute concordance scores between PKC knockdown and chemical perturbations
  - Method: Connectivity Map (CMap) approach or weighted Kolmogorov-Smirnov statistic
  - Rank compounds by signature similarity
- Output: Ranked list of compounds with concordance scores

#### 4. Chemical Similarity & Library Comparison
- Convert LINCS compound IDs to SMILES representations
- Calculate chemical similarity between:
  - Top concordant compounds from Phase 1
  - Curated drug-like library
- Methods: 
  - Tanimoto coefficient using Morgan fingerprints (RDKit)
  - Scaffold similarity analysis
- Filter/expand candidate set based on similarity thresholds

#### 5. AI-Powered Docking with Boltz-2
- Integrate **Boltz2** for structure prediction/refinement
  - Generate PKC structure if experimental unavailable
  - Prepare protein structure (add hydrogens, assign charges)
- Select top 5% of compounds from ceSAR-S scoring
- Perform molecular docking:
  - Ligand preparation: 3D conformer generation (RDKit/OpenBabel)
  - Docking: Boltz2 or fallback to AutoDock Vina/GNINA
  - Scoring: binding affinity prediction

#### 6. Consensus Ranking and Validation
- Combine rankings from ceSAR-S and docking
  - Methods: geometric mean, rank aggregation

#### 7. [References and Further Reading](#references-and-further-reading)
---


## Environment Setup

### Why Docker?

This implementation of ceSAR with Boltz-2 integration involves multiple complex software dependencies with specific version requirements:

- **Python scientific stack**: NumPy, Pandas, SciPy for data manipulation and statistics
- **Cheminformatics tools**: RDKit for molecular representation and fingerprint generation
- **Boltz-2 dependencies**: PyTorch, deep learning libraries, and biomolecular modeling frameworks

Docker containers provide reproducibility by encapsulating all dependencies in an isolated environment. This ensures that the code will run identically on your machine, on a colleague's laptop, or on a cloud computing cluster. No more "works on my machine" problems!

### Prerequisites

Before you begin, ensure you have:

- **Docker**: [Install Docker Desktop](https://docs.docker.com/get-docker/) (version 20.10 or higher)
- **GPU Support** (highly recommended for Boltz-2): 
  - NVIDIA GPU with 24GB+ VRAM (48GB+ recommended for large proteins)
  - CUDA 11.8 or higher
  - [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html) installed
- **System Requirements**:
  - 64GB+ RAM (128GB recommended for large-scale screens)
  - 256GB+ free disk space (for container, datasets, and LINCS signatures)
  - Stable internet connection for initial data downloads

While these are recommended specs, you can run smaller screens on less powerful hardware. Just be aware that performance will vary. If you are interested in reducing cost, consider either building your own PC with a consumer-grade GPU (e.g., NVIDIA RTX 3090/4090) or using cloud services like AWS, GCP, or Azure with GPU instances.

### Setup Steps

#### 1. Clone the Repository

```bash
git clone https://github.com/gabenavarro/MLContainerLab.git
cd MLContainerLab
```

#### 2. Build the Docker Image

Base image: Python 3.10 with CUDA 11.8. PyTorch 2.0+ with CUDA support for Boltz-2. RDKit 2023.09.1 for cheminformatics. Boltz-2 and dependencies installed from source. You can choose any tag you want for the image. Feel free to play around with the base image, just make sure the host has the same or higher CUDA version.

```bash
docker build -f ./assets/build/Dockerfile.lincs.cu126cp310 -t cesar-lincs-boltz2:126-310 .
```

**Build time**: Expect 20-30 minutes depending on your internet connection. The image includes:
- CUDA-enabled PyTorch (several GB)
- Boltz-2 model weights (pre-downloaded to avoid repeated downloads)
- RDKit and molecular visualization tools
- Jupyter Lab with scientific Python stack

#### 3. Run the Docker Container

**For GPU-enabled execution (recommended):**

```bash
docker run -dt \
  --gpus all \
  --shm-size=96g \
  -v "$(pwd):/workspace" \
  --name cesar-lincs-boltz2-container \
  --env NVIDIA_VISIBLE_DEVICES=all \
  cesar-lincs-boltz2:126-310
```

**Flag explanations**:
- `--gpus all`: Grants container access to all available GPUs (requires NVIDIA Container Toolkit)
- `--shm-size=96g`: ADJUST ACCORDINGLY TO YOUR SYSTEM'S RESOURCES. Increases shared memory size for PyTorch DataLoaders and large batches in Boltz-2
- `-v "$(pwd):/workspace"`: Mounts your current directory inside the container, enabling persistent storage and easy file exchange
- `--name cesar-lincs-boltz2-container`: Assigns a memorable name for easy reference
- `-dt`: Runs in detached mode with pseudo-TTY allocation

#### 4. Access Jupyter Lab through VSCode

In this example, we will use Visual Studio Code to access the container. You can use any IDE of your choice.

```bash
# In a scriptable manner
CONTAINER_NAME=cesar-lincs-boltz2-container
FOLDER=/workspace
HEX_CONFIG=$(printf {\"containerName\":\"/$CONTAINER_NAME\"} | od -A n -t x1 | tr -d '[\n\t ]')
code --folder-uri "vscode-remote://attached-container+$HEX_CONFIG$FOLDER"
```

## Data Acquisition and Preprocessing

In [2]:
import os

In [2]:
# Download LINCS datasets to /workspace/datasets/lincs
!mkdir -p /workspace/datasets/lincs
# Check if files already exist to avoid re-downloading; compounds
if not os.path.exists('/workspace/datasets/lincs/cp_predicted_RNAseq_profiles.gctx'):
    !wget -O /workspace/datasets/lincs/cp_predicted_RNAseq_profiles.gctx https://lincs-dcic.s3.amazonaws.com/LINCS-data-2020/RNA-seq/cp_predicted_RNAseq_profiles.gctx
# shRNA
if not os.path.exists('/workspace/datasets/lincs/shRNA_predicted_RNAseq_profiles.gctx'):
    !wget -O /workspace/datasets/lincs/shRNA_predicted_RNAseq_profiles.gctx https://lincs-dcic.s3.amazonaws.com/LINCS-data-2020/RNA-seq/shRNA_predicted_RNAseq_profiles.gctx
# over expression
if not os.path.exists('/workspace/datasets/lincs/oe_predicted_RNAseq_profiles.gctx'):
    !wget -O /workspace/datasets/lincs/oe_predicted_RNAseq_profiles.gctx https://lincs-dcic.s3.amazonaws.com/LINCS-data-2020/RNA-seq/oe_predicted_RNAseq_profiles.gctx
# Controls
if not os.path.exists('/workspace/datasets/lincs/ctl_predicted_RNAseq_profiles.gctx'):
    !wget -O /workspace/datasets/lincs/ctl_predicted_RNAseq_profiles.gctx https://lincs-dcic.s3.amazonaws.com/LINCS-data-2020/RNA-seq/ctl_predicted_RNAseq_profiles.gctx

--2025-10-27 20:27:57--  https://lincs-dcic.s3.amazonaws.com/LINCS-data-2020/RNA-seq/ctl_predicted_RNAseq_profiles.gctx
Resolving lincs-dcic.s3.amazonaws.com (lincs-dcic.s3.amazonaws.com)... 3.5.10.180, 52.217.106.196, 52.217.203.65, ...
Connecting to lincs-dcic.s3.amazonaws.com (lincs-dcic.s3.amazonaws.com)|3.5.10.180|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17898738640 (17G) [application/octet-stream]
Saving to: ‘/workspace/datasets/lincs/ctl_predicted_RNAseq_profiles.gctx’

/workspace/datasets 100%[===================>]  16.67G  13.3MB/s    in 21m 46s 

2025-10-27 20:51:07 (13.1 MB/s) - ‘/workspace/datasets/lincs/ctl_predicted_RNAseq_profiles.gctx’ saved [17898738640/17898738640]



In [ ]:
# Download natural product library to /workspace/datasets/molecules/natural_products
!mkdir -p /workspace/datasets/molecules/natural_products
# MEGx_Release_2025_09_05.7z
if not os.path.exists('/workspace/datasets/molecules/natural_products/MEGx_Release_2025_09_05.7z'):
    !wget -O /workspace/datasets/molecules/natural_products/MEGx_Release_2025_09_05.7z https://ac-discovery.com/wp-content/uploads/MEGx_Release_2025_09_05.7z
    # Extract the 7z file
    !7z x /workspace/datasets/molecules/natural_products/MEGx_Release_2025_09_05.7z -o/workspace/datasets/molecules/natural_products/

# https://ac-discovery.com/wp-content/uploads/NATx_Release_2025_09_05.zip
if not os.path.exists('/workspace/datasets/molecules/natural_products/NATx_Release_2025_09_05.zip'):
    !wget -O /workspace/datasets/molecules/natural_products/NATx_Release_2025_09_05.zip https://ac-discovery.com/wp-content/uploads/NATx_Release_2025_09_05.zip
    # Extract the zip file
    !unzip /workspace/datasets/molecules/natural_products/NATx_Release_2025_09_05.zip -d /workspace/datasets/molecules/natural_products/

# https://ac-discovery.com/wp-content/uploads/MACROx_Release_2025_09_05.7z
if not os.path.exists('/workspace/datasets/molecules/natural_products/MACROx_Release_2025_09_05.7z'):
    !wget -O /workspace/datasets/molecules/natural_products/MACROx_Release_2025_09_05.7z https://ac-discovery.com/wp-content/uploads/MACROx_Release_2025_09_05.7z
    # Extract the 7z file
    !7z x /workspace/datasets/molecules/natural_products/MACROx_Release_2025_09_05.7z -o/workspace/datasets/molecules/natural_products/

In [3]:
# https://www.biosino.org/iMAC/cyclicpepedia/static/data/excel/Bioassay.xlsx
if not os.path.exists('/workspace/datasets/molecules/natural_products/Bioassay.xlsx'):
    !wget -O /workspace/datasets/molecules/natural_products/Bioassay.xlsx https://www.biosino.org/iMAC/cyclicpepedia/static/data/excel/Bioassay.xlsx
# Convert Bioassay.xlsx to Bioassay.csv
if not os.path.exists('/workspace/datasets/molecules/natural_products/Bioassay.csv'):
    import pandas as pd
    df = pd.read_excel('/workspace/datasets/molecules/natural_products/Bioassay.xlsx')
    df.to_csv('/workspace/datasets/molecules/natural_products/Bioassay.csv', index=False)

# https://www.biosino.org/iMAC/cyclicpepedia/static/data/structure_2dmol.zip
if not os.path.exists('/workspace/datasets/molecules/natural_products/structure_2dmol.zip'):
    !wget -O /workspace/datasets/molecules/natural_products/structure_2dmol.zip https://www.biosino.org/iMAC/cyclicpepedia/static/data/structure_2dmol.zip
    # Extract the zip file
    !unzip /workspace/datasets/molecules/natural_products/structure_2dmol.zip -d /workspace/datasets/molecules/natural_products/

In [ ]:
# Download fda-approved drugs to /workspace/datasets/molecules/fda_approved
!mkdir -p /workspace/datasets/molecules/fda_approved
if not os.path.exists('/workspace/datasets/molecules/fda_approved/compounds.parquet'):
    !wget -O /workspace/datasets/molecules/fda_approved/compounds.parquet https://ftp.ebi.ac.uk/pub/databases/chembl/SureChEMBL/bulk_data/latest/compounds.parquet
    !wget -O /workspace/datasets/molecules/fda_approved/fields.parquet https://ftp.ebi.ac.uk/pub/databases/chembl/SureChEMBL/bulk_data/latest/fields.parquet
    !wget -O /workspace/datasets/molecules/fda_approved/patent_compound_map.parquet https://ftp.ebi.ac.uk/pub/databases/chembl/SureChEMBL/bulk_data/latest/patent_compound_map.parquet
    !wget -O /workspace/datasets/molecules/fda_approved/patents.parquet https://ftp.ebi.ac.uk/pub/databases/chembl/SureChEMBL/bulk_data/latest/patents.parquet

## Signature Concordance Analysis

Lets start by loading the LINCS gctx files using our custom `LINCSGCTXReader` parser. We will load both the chemical perturbation and shRNA knockdown datasets. We will take the following approach for our ceSAR analysis:

1. Load LINCS chemical perturbation
2. Load LINCS shRNA knockdown signatures
3. Extract consensus PKC knockdown signature from shRNA dataset
4. Compute concordance scores between PKC knockdown and chemical perturbations
5. Create ranked list of compounds with concordance scores

In [1]:
import sys
sys.path.append('/workspace')
from src.lincs import LINCSGCTXReader, devLINCSGCTXReader

In [2]:
# Initialize the reader for the compound dataset
cp_reader = devLINCSGCTXReader("/workspace/datasets/lincs/cp_predicted_RNAseq_profiles.gctx")
print("\n=== File Information ===")
print(f"Genes: {cp_reader.n_genes}")
print(f"Samples: {cp_reader.n_samples}")
print(f"\nSample metadata columns: {cp_reader.col_meta.columns.tolist()}")
print("\nFirst 5 samples:")
cp_reader.col_meta.head()

Loaded GCTX file: /workspace/datasets/lincs/cp_predicted_RNAseq_profiles.gctx
Shape: (1805898, 23614) (23614 genes × 1805898 samples)
Data type: float32
Estimated size: 170.58 GB

=== File Information ===
Genes: 23614
Samples: 1805898

Sample metadata columns: ['cell', 'dose', 'id', 'inchi_key', 'pert_aliases', 'pert_id', 'pertname', 'smiles', 'timepoint']

First 5 samples:


,cell,dose,id,inchi_key,pert_aliases,pert_id,pertname,smiles,timepoint
0,b'A375',b'2.5 uM',b'ABY001_A375_XH_X1_B15:M04',b'WAEXFXRVDQXREF-UHFFFAOYSA-N',b'nan',b'BRD-K81418486',b'vorinostat',b'ONC(=O)CCCCCCC(=O)Nc1ccccc1',b'3 h'
1,b'A375',b'10 uM',b'ABY001_A375_XH_X1_B15:D18',b'OCKHRKSTDPOHEN-BQYQJAHWSA-N',b'nan',b'BRD-K70511574',b'HMN-214',b'COc1ccc(cc1)S(=O)(=O)N(C(C)=O)c1ccccc1C=Cc1c...,b'24 h'
2,b'A375',b'2.5 uM',b'ABY001_A375_XH_X1_B15:B15',b'JWNPDZNEKVCWMY-VQHVLOKHSA-N',b'nan',b'BRD-K85606544',b'neratinib',b'CCOc1cc2ncc(C#N)c(Nc3ccc(OCc4ccccn4)c(Cl)c3)...,b'24 h'
3,b'A375',b'2.5 uM',b'ABY001_A375_XH_X1_B15:J11',b'JWNPDZNEKVCWMY-VQHVLOKHSA-N',b'nan',b'BRD-K85606544',b'neratinib',b'CCOc1cc2ncc(C#N)c(Nc3ccc(OCc4ccccn4)c(Cl)c3)...,b'3 h'
4,b'A375',b'10 uM',b'ABY001_A375_XH_X1_B15:P09',b'AYUNIORJHRXIBJ-ZGQRYRSUSA-N',b'nan',b'BRD-A61304759',b'tanespimycin',b'COC1CC(C)CC2=C(NCC=C)C(=O)C=C(NC(=O)C(C)=CC=...,b'3 h'


In [3]:
# Initialize reader for shRNA dataset
sh_reader = LINCSGCTXReader('/workspace/datasets/lincs/shRNA_predicted_RNAseq_profiles.gctx')
print("\n=== File Information ===")
print(f"Genes: {sh_reader.n_genes}")
print(f"Samples: {sh_reader.n_samples}")
print(f"\nSample metadata columns: {sh_reader.col_meta.columns.tolist()}")
print("\nFirst 5 samples:")
sh_reader.col_meta.head()

Loaded GCTX file: /workspace/datasets/lincs/shRNA_predicted_RNAseq_profiles.gctx
Shape: (453175, 23614) (23614 genes × 453175 samples)
Data type: float32
Estimated size: 42.81 GB

=== File Information ===
Genes: 23614
Samples: 453175

Sample metadata columns: ['cell', 'dose', 'id', 'pertname', 'timepoint']

First 5 samples:


,cell,dose,id,pertname,timepoint
0,b'A375',b'nan',b'DER001_A375_96H_X1_B7_DUO52HI53LO:G13',b'GNE',b'96 h'
1,b'A375',b'nan',b'DER001_A375_96H_X2_B7_DUO52HI53LO:A11',b'EIF2AK3',b'96 h'
2,b'A375',b'nan',b'DER001_A375_96H_X3_B7_DUO52HI53LO:P16',b'NPTN',b'96 h'
3,b'A375',b'nan',b'DER001_A375_96H_X2_B7_DUO52HI53LO:E01',b'PTP4A1',b'96 h'
4,b'A375',b'nan',b'DER001_A375_96H_X1_B7_DUO52HI53LO:L04',b'AKR1C2',b'96 h'


In [4]:
pkc_samples = sh_reader.search_perturbagens('PRKCB', field='pertname')
print(f"Identified PKC-related samples: {pkc_samples['pertname'].unique().tolist()}")
print(f"Cell types tested: {pkc_samples['cell'].unique().tolist()}")
print(f"Unique timepoints: {pkc_samples['timepoint'].unique().tolist()}")
pkc_samples.head(5)

Identified PKC-related samples: [b'PRKCB']
Cell types tested: [b'VCAP', b'A375', b'A549', b'HA1E', b'HCC515', b'HEPG2', b'MCF7', b'PC3', b'HEKTE', b'SW480']
Unique timepoints: [b'120 h', b'96 h', b'nan']


,cell,dose,id,pertname,timepoint
19383,b'VCAP',b'nan',b'ERGK006_VCAP_120H_X2.A2_B4_DUO52HI53LO:K23',b'PRKCB',b'120 h'
19493,b'VCAP',b'nan',b'ERGK006_VCAP_120H_X2.A2_B4_DUO52HI53LO:O15',b'PRKCB',b'120 h'
19524,b'VCAP',b'nan',b'ERGK006_VCAP_120H_X2.A2_B4_DUO52HI53LO:M01',b'PRKCB',b'120 h'
19539,b'VCAP',b'nan',b'ERGK006_VCAP_120H_X3_B2_DUO52HI53LO:O15',b'PRKCB',b'120 h'
19616,b'VCAP',b'nan',b'ERGK006_VCAP_120H_X3_B2_DUO52HI53LO:I13',b'PRKCB',b'120 h'


In [5]:
# Create a consensus knockdown signature for PRKCB or PRKCZ
prkcz_signature = sh_reader.extract_knockdown_consensus(
    'PRKCZ',
    cell_lines=[b'VCAP'],
    timepoints=[b'120 h'],
    consensus_type=None
)

# Now search for perturbagens that target PRKCZ
results_df = cp_reader.compute_signature_concordance(
    prkcz_signature, 
    threshold=0.5,
    cell_lines=[b'VCAP'],
    timepoints=[b'6 h', b'24 h'],
    chunk_size=5000
)
results_df.sort_values('bai_score', ascending=False).head(10)

Processing chunks: 100%|██████████| 5/5 [05:21<00:00, 64.36s/it] 


,cell,dose,id,inchi_key,pert_aliases,pert_id,pertname,smiles,timepoint,n_replicates,replicate_ids,bai_score,lower_credible,upper_credible,mean_agreement
4532,b'VCAP',b'10 uM',b'CPC003_VCAP_6H_X5_F1B5_DUO52HI53LO:H13',b'MKXZASYAUGDDCJ-NJAFHUGGSA-N',b'dextromethorphan',b'BRD-K33211335',b'BRD-K33211335',b'COc1ccc2C[C@H]3[C@H]4CCCC[C@@]4(CCN3C)c2c1',b'6 h',9,"CPC003_VCAP_6H_X5_F1B5_DUO52HI53LO:H13,CPC003_...",0.861763,0.857361,0.866165,0.861794
6517,b'VCAP',b'10 uM',b'CPC005_VCAP_24H_X4_F1B5_DUO52HI53LO:F07',b'RSUVYMGADVXGOU-BUHFOSPRSA-N',b'nan',b'BRD-K40901640',b'cinanserin',b'CN(C)CCCSc1ccccc1NC(=O)C=Cc1ccccc1',b'24 h',10,"CPC005_VCAP_24H_X4_F1B5_DUO52HI53LO:F07,CPC005...",0.843698,0.839067,0.848329,0.843727
5243,b'VCAP',b'10 uM',b'CPC001_VCAP_24H_X3_B3_DUO52HI53LO:P11',b'OHQQFYCZENVXDJ-UHFFFAOYSA-N',b'JAK3-inhibitor-I',b'BRD-K72541103',b'BRD-K72541103',b'OCc1cc2ncnc(Nc3ccc(O)cc3)c2cc1CO',b'24 h',8,"CPC001_VCAP_24H_X3_B3_DUO52HI53LO:P11,CPC001_V...",0.841391,0.836732,0.846050,0.841420
8505,b'VCAP',b'10 uM',b'CPC001_VCAP_24H_X1_B3_DUO52HI53LO:M15',b'XJJZQXUGLLXTHO-UHFFFAOYSA-N',b'nan',b'BRD-A49370193',b'Ro-60-0175',b'CC(N)Cn1ccc2cc(F)c(Cl)cc12',b'24 h',8,"CPC001_VCAP_24H_X1_B3_DUO52HI53LO:M15,CPC001_V...",0.840981,0.836317,0.845645,0.841010
2142,b'VCAP',b'10 uM',b'CPC005_VCAP_24H_X2_B3_DUO52HI53LO:G14',b'GGUSQTSTQSHJAH-UHFFFAOYSA-N',b'nan',b'BRD-A61392169',b'eliprodil',b'OC(CN1CCC(Cc2ccc(F)cc2)CC1)c1ccc(Cl)cc1',b'24 h',8,"CPC005_VCAP_24H_X2_B3_DUO52HI53LO:G14,CPC005_V...",0.838603,0.833911,0.843295,0.838632
8413,b'VCAP',b'10 uM',b'CPC001_VCAP_24H_X4_F1B5_DUO52HI53LO:P12',b'XEYBRNLFEZDVAW-ARSRFYASSA-N',b'nan',b'BRD-K26521938',b'dinoprostone',b'CCCCC[C@H](O)C=C[C@H]1[C@H](O)CC(=O)[C@@H]1C...,b'24 h',8,"CPC001_VCAP_24H_X4_F1B5_DUO52HI53LO:P12,CPC001...",0.838114,0.833416,0.842812,0.838143
5664,b'VCAP',b'10 uM',b'CPC005_VCAP_24H_X2_B3_DUO52HI53LO:O07',b'PLDUPXSUYLZYBN-UHFFFAOYSA-N',b'nan',b'BRD-K55127134',b'fluphenazine',b'OCCN1CCN(CCCN2c3ccccc3Sc3ccc(cc23)C(F)(F)F)CC1',b'24 h',17,"CPC005_VCAP_24H_X2_B3_DUO52HI53LO:O07,CPC005_V...",0.837897,0.833196,0.842597,0.837925
5702,b'VCAP',b'10 uM',b'CPC005_VCAP_24H_X2_B3_DUO52HI53LO:I10',b'PMKJGGBYYNEYPA-UHFFFAOYSA-N',b'htmt',b'BRD-A66435872',b'HTMT',b'CC(CCCCC(=O)Nc1ccc(cc1)C(F)(F)F)NCCc1c[nH]cn1',b'24 h',9,"CPC005_VCAP_24H_X2_B3_DUO52HI53LO:I10,CPC005_V...",0.837185,0.832476,0.841894,0.837214
8478,b'VCAP',b'10 uM',b'CPC003_VCAP_6H_X5_F1B5_DUO52HI53LO:J07',b'XHLOUFPZLUULGI-UHFFFAOYSA-N',b'nan',b'BRD-K37883585',b'BRD-K37883585',b'COc1ccc2[nH]cc(CCNCc3ccc(Br)cc3)c2c1',b'6 h',5,"CPC003_VCAP_6H_X5_F1B5_DUO52HI53LO:J07,CPC003_...",0.836638,0.831923,0.841353,0.836667
1,b'VCAP',b'10 uM',b'CPC007_VCAP_24H_X2_B3_DUO52HI53LO:G09',b'AACFPJSJOWQNBN-UHFFFAOYSA-N',b'nan',b'BRD-K03568209',b'MW-STK33-4C',b'Oc1ccc2oc3C(=O)NCCCc3c2c1',b'24 h',15,"CPC007_VCAP_24H_X2_B3_DUO52HI53LO:G09,CPC007_V...",0.834533,0.829793,0.839272,0.834561


In [6]:
print(f"Number of positively concordant perturbagens: {len(results_df)}")
print(f"Unique smiles count above 0.8: {len(results_df[results_df['bai_score']>0.6]['smiles'].unique())}")
print(f"Unique smiles: {results_df['smiles'].nunique()}")
# results_df.to_parquet('/workspace/datasets/lincs/outputs/prkcz_concordant_perturbagens.parquet', index=False)

Number of positively concordant perturbagens: 26487
Unique smiles count above 0.8: 8618
Unique smiles: 15103


In [ ]:
# Create a consensus knockdown signature for PRKCB or PRKCZ
prkcb_signature = sh_reader.extract_knockdown_consensus(
    'PRKCB',
    cell_lines=[b'VCAP'],
    timepoints=[b'120 h'],
    consensus_type=None
)
prkcb_signature.shape

In [ ]:
# Now search for perturbagens that target PRKCB
results_df = cp_reader.compute_signature_concordance(
    prkcb_signature, 
    threshold=0.5,
    cell_lines=[b'VCAP'],
    timepoints=[b'6 h', b'24 h'],
)
results_df.sort_values('bai_score', ascending=False).head(10)

In [ ]:
print(f"Number of positively concordant perturbagens: {len(results_df)}")
print(f"Unique smiles count above 0.9: {len(results_df[results_df['bai_score']>0.9]['smiles'].unique())}")
print(f"Unique smiles: {results_df['smiles'].nunique()}")
results_df.to_parquet('/workspace/datasets/lincs/outputs/prkcb_concordant_perturbagens.parquet', index=False)

## References and Further Reading

### Seminal Papers

- **Connectivity Map** (2006): [Lamb et al., Science](https://doi.org/10.1126/science.1132939) - The foundational work establishing transcriptional signature connectivity for drug discovery

- **LINCS Program** (2017): [Subramanian et al., Cell](https://doi.org/10.1016/j.cell.2017.10.049) - Introduction of the Library of Integrated Network-based Cellular Signatures with 1 million profiles

- **iLINCS Platform** (2022): [Pilarczyk et al., Nature Communications](https://doi.org/10.1038/s41467-022-32205-3) - The computational infrastructure enabling ceSAR's signature connectivity analysis

- **DUD-E Benchmark** (2012): [Mysinger et al., J. Med. Chem.](https://doi.org/10.1021/jm300687e) - The gold standard benchmark for evaluating virtual screening methods

### Recent Developments

- **Boltz-2 and AI Docking**: [Biomolecular structure prediction with deep learning](https://github.com/jwohlwend/boltz) - Modern transformer-based approaches to protein-ligand docking

- **Deep Learning for Drug Discovery** (2023): [Li et al., Med Review](https://doi.org/10.1515/mr-2023-0023) - Comprehensive overview of AI methods predicting compound-protein interactions

- **Virtual Screening Evolution** (2023): [Lyu et al., Nature Chemical Biology](https://doi.org/10.1038/s41589-023-01281-8) - Modern perspectives on expanding virtual screening libraries

### Implementations and Tools

- **ceSAR Official Implementation**: [sig2lead GitHub](https://github.com/sig2lead) - Original R Shiny app and standalone application

- **sig2lead Web Server**: [http://sig2lead.net](http://sig2lead.net) - Interactive interface for ceSAR-S without local installation

- **iLINCS API**: [https://www.ilincs.org](https://www.ilincs.org) - Access to LINCS signatures and concordance computation

- **ChEMBL Database**: [https://www.ebi.ac.uk/chembl/](https://www.ebi.ac.uk/chembl/) - Bioactivity database for training and validation

- **RDKit Documentation**: [https://www.rdkit.org/docs/](https://www.rdkit.org/docs/) - Comprehensive cheminformatics toolkit

### Tutorials and Educational Resources

- **Drug Discovery with Python**: [Practical Cheminformatics](https://www.oreilly.com/library/view/deep-learning-for/9781492039822/) - O'Reilly book covering molecular representations and ML

- **LINCS Tutorials**: [BD2K-LINCS DCIC Training](http://lincs-dcic.org/#/tutorials) - Video tutorials on using LINCS resources

- **Structure-Based Drug Design**: [TeachOpenCADD](https://github.com/volkamerlab/teachopencadd) - Jupyter-based tutorials for computational drug design

- **Signature Connectivity Analysis**: [clue.io Resources](https://clue.io) - Broad Institute's tools and tutorials for connectivity mapping